# ClickHouse_Superset

##  Установка библиотек в Jupyter

In [6]:
!pip install clickhouse-connect pandas pyarrow

In [7]:
import clickhouse_connect

# 1. Подключаюсь к локальной базе в Docker
client = clickhouse_connect.get_client(
    host='localhost', 
    port=8123, 
    username='default', 
    password='mysecretpassword'
)
print("Успешно подключились к домашнему ClickHouse!")

# 2. Удаляю старую таблицу, если она была
client.command('DROP TABLE IF EXISTS fake_sales')

# 3. Создаю таблицу для аналитики интернет-магазина
client.command('''
CREATE TABLE fake_sales (
    user_id UInt32,
    category String,
    price Float64,
    timestamp DateTime
) ENGINE = MergeTree()
ORDER BY (category, timestamp)
''')

print("Генерю 5 000 000 строк")

# Функция numbers() мгновенно создает миллионы чисел, а rand() превращает их в случайные покупки
client.command('''
INSERT INTO fake_sales
SELECT 
    rand() % 100000 + 1 AS user_id,
    ['Электроника', 'Одежда', 'Книги', 'Дом', 'Спорт'][rand() % 5 + 1] AS category,
    (rand() % 10000 + 100) / 10 AS price,
    now() - rand() % 864000 AS timestamp
FROM numbers(5000000)
''')

# 4. Проверяю результат подсчетом строк
count = client.command('SELECT count() FROM fake_sales')
print(f"\n Создана таблица 'fake_sales'.")
print(f"Всего реальных строк в локальной базе: {count:,}")


Успешно подключились к домашнему ClickHouse!
Генерю 5 000 000 строк

 Создана таблица 'fake_sales'.
Всего реальных строк в локальной базе: 5,000,000
